# Hands-on Sesi 14 — Building Batch Data Pipelines with Spark
**Topik:** Membaca CSV → Transformasi → Validasi → Agregasi → Menyimpan Parquet

Notebook ini dirancang sebagai latihan praktik untuk kelas Data Engineering. Peserta akan membangun pipeline transaksi menggunakan PySpark, lalu membandingkan strategi penyimpanan `overwrite` dan `append`.

**Tujuan pembelajaran**
1. Membaca CSV menjadi Spark DataFrame.
2. Memeriksa skema dan contoh data.
3. Melakukan filter dan membuat kolom turunan.
4. Memvalidasi nilai kosong, duplikasi, dan jumlah record.
5. Menghitung metrik per kota.
6. Menulis dan membaca kembali data Parquet.
7. Memahami risiko `overwrite` dan `append`.

**Cara menjalankan**
- Jalankan sel dari atas ke bawah.
- Notebook dapat dijalankan di Google Colab atau lingkungan yang sudah memiliki Java dan PySpark.
- Jika menjalankan di Colab, aktifkan sel instalasi berikut terlebih dahulu.

## 0. Menyiapkan lingkungan

Jika PySpark belum terpasang, jalankan sel berikut. Di lingkungan lokal yang sudah siap, sel ini bisa dilewati.

In [ ]:
# Google Colab / lingkungan Python yang belum memiliki PySpark:
%pip -q install pyspark

## 1. Membuat SparkSession
Kita membuat sesi Spark sebagai titik masuk untuk membaca dan memproses DataFrame.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, when

spark = (
    SparkSession.builder
    .appName("HandsOnBatchPipelineSesi14")
    .master("local[*]")
    .getOrCreate()
)

spark

## 2. Membuat dataset transaksi latihan
Agar praktik dapat langsung berjalan, dataset contoh dibuat di notebook dan disimpan sebagai CSV. Dataset sengaja berisi status transaksi yang berbeda, satu nilai kosong, serta satu ID transaksi duplikat untuk latihan validasi.

In [ ]:
import csv
from pathlib import Path

data_dir = Path("data_sesi14")
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / "transaksi.csv"

rows = [
    ["TR001", "2026-10-01", "Jakarta", "Selesai", 100000],
    ["TR002", "2026-10-01", "Jakarta", "Selesai", 200000],
    ["TR003", "2026-10-01", "Bandung", "Selesai", 150000],
    ["TR004", "2026-10-02", "Bandung", "Pending", 50000],
    ["TR005", "2026-10-02", "Surabaya", "Selesai", 300000],
    ["TR006", "2026-10-02", "Jakarta", "Batal", 75000],
    ["TR007", "2026-10-03", "Surabaya", "Selesai", None],
    ["TR008", "2026-10-03", "Bandung", "Selesai", 125000],
    ["TR009", "2026-10-03", "Jakarta", "Selesai", 90000],
    ["TR009", "2026-10-03", "Jakarta", "Selesai", 90000],  # ID duplikat untuk latihan
]

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id_transaksi", "tanggal", "kota", "status", "nilai"])
    writer.writerows(rows)

print(f"Dataset dibuat: {csv_path.resolve()}")

### Eksplorasi awal
Jalankan sel berikut, lalu perhatikan nama kolom, tipe data, lima baris pertama, dan jumlah record.

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(csv_path))
)

df.printSchema()
df.show(5, truncate=False)
print("Jumlah record awal:", df.count())

**Pertanyaan 1**
- Kolom mana yang bertipe numerik?
- Berapa jumlah record awal?
- Mengapa `inferSchema=True` dapat membantu, dan mengapa skema eksplisit bisa lebih aman untuk pipeline produksi?

Tulis jawaban Anda pada sel Markdown berikut.

**Jawaban peserta:**

- Tipe data numerik:
- Jumlah record awal:
- Catatan tentang inferSchema:

## 3. Transformasi data
Kita memilih kolom yang diperlukan, mempertahankan transaksi berstatus `Selesai`, dan membuang baris yang nilai transaksinya kosong. Setelah itu, kita membuat kolom ilustratif `nilai_plus_pajak` dengan pengali 1,11. Ini hanya contoh perhitungan, bukan panduan pajak.

In [ ]:
clean = (
    df
    .select("id_transaksi", "tanggal", "kota", "status", "nilai")
    .filter(col("status") == "Selesai")
    .filter(col("nilai").isNotNull())
    .withColumn("nilai_plus_pajak", col("nilai") * 1.11)
)

clean.show(truncate=False)
print("Jumlah record setelah filter:", clean.count())

**Latihan 2 — Modifikasi kode**
1. Ubah filter agar hanya mengambil transaksi dengan `nilai >= 100000`.
2. Tambahkan kolom `kategori_nilai`: `Besar` jika nilai minimal 200000, selain itu `Kecil`.
3. Bandingkan jumlah baris sebelum dan sesudah perubahan.

Gunakan sel berikut untuk mencoba.

In [ ]:
# TODO: lengkapi atau modifikasi kode latihan di bawah ini

# latihan = (
#     df
#     ...
# )

# latihan.show(truncate=False)

## 4. Validasi kualitas data
Jangan langsung menyimpan hasil sebelum memeriksa skema, nilai kosong, duplikasi, dan jumlah record. Kita memeriksa nilai kosong pada data awal karena tahap transformasi di atas sudah membuang nilai kosong.

In [ ]:
print("Skema data awal:")
df.printSchema()

null_nilai = df.filter(col("nilai").isNull()).count()
print("Record dengan nilai null pada data awal:", null_nilai)

print("Record data awal:", df.count())
print("Record setelah filter status selesai:", df.filter(col("status") == "Selesai").count())
print("Record setelah filter status dan nilai tidak null:", clean.count())

print("ID transaksi yang muncul lebih dari sekali:")
(
    df.groupBy("id_transaksi")
    .agg(count("*").alias("jumlah kemunculan"))
    .filter(col("jumlah kemunculan") > 1)
    .show(truncate=False)
)

### Diskusi: apa yang dilakukan terhadap duplikasi?
ID duplikat tidak otomatis berarti seluruh baris harus dihapus. Pastikan dulu aturan bisnis: apakah `id_transaksi` memang unik? Pada latihan ini kita akan membuat satu versi yang menghapus duplikasi berdasarkan `id_transaksi`, lalu membandingkan jumlah recordnya.

In [ ]:
clean_dedup = clean.dropDuplicates(["id_transaksi"])

print("Record sebelum deduplikasi:", clean.count())
print("Record sesudah deduplikasi:", clean_dedup.count())
clean_dedup.show(truncate=False)

**Pertanyaan 3**
- Berapa baris yang memiliki `nilai` null?
- ID mana yang duplikat?
- Mengapa deduplikasi sebaiknya didasarkan pada key bisnis yang benar, bukan sembarang kolom?
- Dalam kondisi apa dua record dengan nilai yang sama justru merupakan transaksi berbeda?

## 5. Agregasi per kota
Sekarang kita menghitung jumlah transaksi dan total nilai per kota. Di bagian ini kita menggunakan data yang sudah difilter dan dideduplikasi agar contoh tidak menghitung ID duplikat dua kali.

In [ ]:
ringkasan = (
    clean_dedup
    .groupBy("kota")
    .agg(
        count("*").alias("jumlah_transaksi"),
        sum("nilai").alias("total_nilai"),
        avg("nilai").alias("rata_rata_nilai")
    )
)

ringkasan.orderBy("kota").show(truncate=False)

**Latihan 4 — Tambahkan metrik**
Tambahkan metrik nilai transaksi terbesar (`max`) dan terkecil (`min`) untuk setiap kota. Import fungsi yang diperlukan terlebih dahulu.

In [ ]:
# TODO:
# from pyspark.sql.functions import max, min
# ringkasan_lengkap = ...
# ringkasan_lengkap.show(truncate=False)

## 6. Menyimpan ke Parquet dan membaca kembali
Parquet merupakan format kolumnar yang banyak digunakan untuk data analitik. Kita menyimpan ringkasan, membaca kembali, lalu memeriksa hasilnya.

In [ ]:
output_path = "output_sesi14/ringkasan_kota"

(
    ringkasan
    .write
    .mode("overwrite")
    .parquet(output_path)
)

hasil_baca = spark.read.parquet(output_path)
hasil_baca.orderBy("kota").show(truncate=False)
print("Jumlah baris ringkasan:", hasil_baca.count())

## 7. Hands-on: memahami `overwrite` vs `append`
### A. Simulasi `overwrite`
Mode `overwrite` mengganti data yang sudah ada pada lokasi target yang ditulis. Gunakan lokasi latihan khusus, bukan direktori data penting.

In [ ]:
demo_path = "output_sesi14/demo_mode"

batch_1 = spark.createDataFrame(
    [("TR001", "Jakarta", 100000), ("TR002", "Bandung", 150000)],
    ["id_transaksi", "kota", "nilai"]
)

batch_2 = spark.createDataFrame(
    [("TR003", "Surabaya", 200000)],
    ["id_transaksi", "kota", "nilai"]
)

batch_1.write.mode("overwrite").parquet(demo_path)
print("Setelah menulis batch_1:")
spark.read.parquet(demo_path).show()

batch_2.write.mode("overwrite").parquet(demo_path)
print("Setelah batch_2 ditulis dengan overwrite — batch_1 sudah tergantikan:")
spark.read.parquet(demo_path).show()

### B. Simulasi `append`
Mode `append` menambahkan baris ke output yang sudah ada. Mode ini tidak otomatis mendeteksi transaksi yang sama atau mencegah batch yang sama dimuat dua kali.

In [ ]:
append_path = "output_sesi14/demo_append"

batch_1.write.mode("overwrite").parquet(append_path)
batch_2.write.mode("append").parquet(append_path)

print("Setelah batch_1 lalu batch_2 dengan append:")
spark.read.parquet(append_path).orderBy("id_transaksi").show()

### C. Demonstrasi risiko duplikasi
Jalankan batch yang sama dua kali dengan `append`. Perhatikan bahwa Spark akan menambahkan record yang sama lagi.

In [ ]:
batch_2.write.mode("append").parquet(append_path)

append_result = spark.read.parquet(append_path)
append_result.orderBy("id_transaksi").show()
print("Jumlah record setelah batch_2 di-append dua kali:", append_result.count())

### Panduan memilih strategi
- Pilih **overwrite** jika output memang harus diganti dengan hasil terbaru dari keseluruhan proses, dan penggantian tersebut aman.
- Pilih **append** jika batch baru harus ditambahkan ke data historis.
- Jika batch bisa dijalankan ulang, append perlu dilengkapi kontrol duplikasi, pencatatan batch, atau strategi pemrosesan ulang partisi.
- Untuk data per tanggal/bulan, pertimbangkan menulis ulang hanya partisi periode terkait bila sistem penyimpanan dan desain pipeline mendukungnya.

**Pertanyaan 5**
1. Pada simulasi di atas, berapa record yang tersisa setelah batch_2 ditulis menggunakan `overwrite`?
2. Berapa record setelah batch_1 dan batch_2 ditulis menggunakan `append`?
3. Apa yang terjadi ketika batch_2 di-append dua kali?
4. Untuk ringkasan penjualan seluruh kota yang dihitung ulang setiap malam, mode mana yang lebih masuk akal? Mengapa?
5. Untuk tabel transaksi historis yang menerima transaksi baru setiap hari, mode mana yang dapat digunakan dan kontrol apa yang perlu ditambahkan?

## 8. Tugas akhir — bangun pipeline sendiri
Buat pipeline lengkap yang memenuhi kriteria berikut:
1. Membaca CSV.
2. Menampilkan skema dan lima baris.
3. Memfilter status `Selesai`.
4. Menangani nilai kosong sesuai keputusan yang dijelaskan.
5. Memeriksa ID duplikat dan menerapkan aturan deduplikasi yang tepat.
6. Menghitung jumlah transaksi, total nilai, dan rata-rata nilai per kota.
7. Menyimpan hasil ke Parquet.
8. Membaca kembali output dan memvalidasi hasilnya.

### Kriteria keberhasilan
- Kode berjalan dari awal hingga akhir.
- Jumlah record sebelum/sesudah filter dijelaskan.
- Keputusan penanganan null dan duplikasi dapat dipertanggungjawabkan.
- Output agregasi masuk akal.
- File Parquet dapat dibaca kembali.
- Pilihan `overwrite` atau `append` disertai alasan.

## 9. Ringkasan
Alur pipeline batch yang dipraktikkan:

**READ → TRANSFORM → VALIDATE → AGGREGATE → WRITE → VERIFY**

Simpan hasil notebook setelah latihan agar kode dan jawaban diskusi dapat ditinjau kembali.